# 2. Federated training
Federated averaging across three banks, compared with each bank alone and with an unsafe centralised model.

In [ ]:
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make core/ importable from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.facecolor": "#0A1830", "axes.facecolor": "#12264A", "axes.edgecolor": "#2A4170",
    "text.color": "#E8EFFA", "axes.labelcolor": "#E8EFFA", "xtick.color": "#8FA3C4", "ytick.color": "#8FA3C4",
    "axes.prop_cycle": plt.cycler(color=["#34D3C0", "#6EA8FF", "#FFB547", "#B58CFF"]), "axes.grid": True, "grid.color": "#2A4170"})
from core.data_gen import load_all, FEATURES
banks, test = load_all()

In [ ]:
from core.federated import run_federation
EPS, ROUNDS = 5, 10
res = run_federation(banks, test, epsilon=EPS, rounds=ROUNDS)
h = pd.DataFrame(res['history']); h.tail(3)

In [ ]:
plt.figure(figsize=(9, 4.2)); plt.plot(h['round'], h['auc'], lw=3, label='Federated')
for k, v in res['local'].items(): plt.axhline(v['auc'], ls=':', label=f'{k} alone')
plt.axhline(res['central']['auc'], ls='--', color='#FFB547', label='Centralised (no privacy)')
plt.xlabel('round'); plt.ylabel('test AUC'); plt.legend(); plt.show()

## Privacy versus accuracy

In [ ]:
from core.federated import epsilon_sweep
s = pd.DataFrame(epsilon_sweep(banks, test)); s['eps'] = s['epsilon'].astype(str)
plt.figure(figsize=(8, 4)); plt.plot(s['eps'], s['auc'], 'o-', label='AUC'); plt.plot(s['eps'], s['recall'], 'o-', label='Recall')
plt.xlabel('epsilon (inf = no noise)'); plt.legend(); plt.show(); s.round(3)